In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler
from reanimator.labelers import TopicChunkPair, calculate_cohens_kappa
load_dotenv()

import nltk
nltk.download('punkt_tab')

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/workspace/src/reanimator/sources.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


In [4]:
t1_doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == "1"]
len(t1_doc_ids)

1647

In [5]:
docs = reanimator.load_documents(doc_ids=t1_doc_ids)[:5]
reanimator.download_documents(docs)

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

accelerator_options = AcceleratorOptions(
        num_threads=8, device=AcceleratorDevice.MPS
    )

Step 1: Loading documents from source...


cord19/trec-covid documents: 100%|██████████| 192509/192509 [00:01<00:00, 170817.23it/s]



Step 2: Fetching URLs and downloading PDFs...
All DOIs already have cached URLs.


Failed to download 10.1093/nar/gkq089: 403 Client Error: Forbidden for url: https://academic.oup.com/nar/article-pdf/38/9/e111/33236399/gkq089.pdf
PDF downloading complete.


In [6]:
#docs = reanimator.load_documents(doc_ids=t1_doc_ids)[:20]
#reanimator.download_documents(docs)
#reanimator.extract_content(docs, accelerator_options)
#reanimator.save_documents(docs, "/workspace/data/documents")

In [7]:
docs = reanimator.load_documents_from_file("/workspace/data/documents")
chunks = reanimator.chunker.chunk(docs)
#chunks = reanimator.chunker.chunk(docs, metadata_fields_to_chunk=["abstract"])

Attempting to load documents from 20 files...
Successfully loaded 20 documents.


In [8]:
table_chunks = [c for c in chunks if c.modality == "table"]
text_chunks = [c for c in chunks if c.modality == "text"]

print(f"{len(table_chunks)} table chunks")
print(f"{len(text_chunks)} text chunks")

29 table chunks
2558 text chunks


In [9]:
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion

In [10]:
indexer_both = Indexer(index_type="bm25", path="/workspace/data/indices/bm25_both/", max_docs=200)
indexer_table = Indexer(index_type="bm25", path="/workspace/data/indices/bm25_table/", max_docs=200)
indexer_text = Indexer(index_type="bm25", path="/workspace/data/indices/bm25_text/", max_docs=200)

indexer_both.index(chunks)
indexer_table.index(table_chunks)
indexer_text.index(text_chunks)

Indexes already exist. Loading from disk.
Indexes already exist. Loading from disk.
Indexes already exist. Loading from disk.


In [11]:
retriever_both = Retriever(indexer=indexer_both)
retriever_table = Retriever(indexer=indexer_table)
retriever_text = Retriever(indexer=indexer_text)

In [12]:
res_both = retriever_both.retrieve(query=topics[0].query_text)
res_table = retriever_table.retrieve(query=topics[0].query_text)
res_text = retriever_text.retrieve(query=topics[0].query_text)

pool = set(reciprocal_rank_fusion([res_both, res_table, res_text]))

In [13]:
#labeler = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"))

In [17]:
labeler = LocalModelLabeler(model="qwen/qwen3-30b-a3b", 
                            base_url="http://192.168.178.179:1234/v1", 
                            concurrency=10,
                            thinking=False)

INFO: LocalModelLabeler initialized with model: qwen/qwen3-30b-a3b at http://192.168.178.179:1234/v1


In [18]:
batch = [TopicChunkPair(topic=topics[0], chunk=chunk) for chunk in chunks if chunk.chunk_id in pool]
len(batch)

236

In [19]:
machine_judgements = await labeler.label_all(batch)

APIConnectionError: Connection error.

In [ ]:
from reanimator.models import save_judgements
model = model.replace("/", "_")
save_judgements(machine_judgements, f"/workspace/data/judgments/machine_{model}_judgements.json")

In [ ]:
#save_judgements(human_judgements, "/workspace/data/judgments/human_judgements.json")

In [ ]:
calculate_cohens_kappa("/workspace/data/judgments/human_judgements.json", "/workspace/data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
import pyterrier as pt

In [ ]:
chunks